# Merge 4 LLMs (Gemini 3.5 Flash, GPT5-mini, Claude 3.5 Haiku, and Kimi K2) and human rewritings with GPT5-mini

### Import Libraries

In [1]:
import numpy as np
import pandas as pd

import os
import json

from google.colab import drive, userdata

### Set up Google Drive mounting and define input/output paths:

In [2]:
# Mount the google drive folder
drive.mount('/content/drive', force_remount = True) # force reconnection

# Define directory, input, and output paths
project_directory = "/content/drive/MyDrive/Colab Notebooks/DS266/final_project"

# Define the paths for all 5 files with rewritten columns
gemini_rw_file = os.path.join(project_directory, "data/gpt data/gpt5mini_gemini.csv")
gpt_rw_file = os.path.join(project_directory, "data/gpt data/gpt5mini_gpt.csv")
claude_rw_file = os.path.join(project_directory, "data/gpt data/gpt5mini_claude.csv")
human_rw_file = os.path.join(project_directory, "data/gpt data/gpt5mini_human.csv")
kimi_rw_file = os.path.join(project_directory, "data/gpt data/gpt5mini_kimi.csv")

# Input file with all Claude rewrittings
input_file = os.path.join(project_directory, "data/final_claude_rewrites.parquet")
# Output file with Claude and GPT rewrittings
output_file = os.path.join(project_directory, "data/final_claude_gpt_rewrites.parquet")

# Ensure the output directory exists
if not os.path.exists(project_directory):
  os.makedirs(project_directory)

Mounted at /content/drive


### Read the input file:

In [3]:
# Read Rahil's Cladue rewritings (Parquet file)
df_all_claude = pd.read_parquet(input_file)
print('Sucessfully read the input file with the questions and Claude rewritings!')

print(f'The input file contains {len(df_all_claude)} records/rows.')
print(f'Column names: {list(df_all_claude.columns)}')

print('\nThe first 5 rows:')
display(df_all_claude.head())

Sucessfully read the input file with the questions and Claude rewritings!
The input file contains 5306 records/rows.
Column names: ['question', 'source', 'human_answers', 'gemini_answers', 'gpt5mini_answer', 'claude_haiku_answers', 'kimi_k2_answers', 'gemini_to_claude', 'gpt_to_claude', 'claude_to_claude', 'human_to_claude', 'kimi_to_claude']

The first 5 rows:


,question,source,human_answers,gemini_answers,gpt5mini_answer,claude_haiku_answers,kimi_k2_answers,gemini_to_claude,gpt_to_claude,claude_to_claude,human_to_claude,kimi_to_claude
0,what composer used sound mass,open_qa,"Composers and works include Barbara Kolb , Pau...",The most prominent composer strongly associate...,"Several 20th-century composers used the ""sound...",# Composers Who Used Sound Mass\n\n**Gyorgy Li...,"The term **""sound mass""** (or **""sound-mass co...",The most prominent composer associated with th...,"Several 20th-century composers pioneered the ""...",# Composers Who Used Sound Mass\n\n**György Li...,Composers employing sound mass techniques incl...,"The term **""sound mass""** (or **""sound-mass co..."
1,where did the persian war take place,open_qa,The Greco-Persian Wars (also often called the ...,The Persian Wars primarily took place in **mai...,If you mean the Greco–Persian Wars (often call...,# The Persian Wars\n\nThe Persian Wars took pl...,The **Persian Wars** (also called the **Greco-...,The Persian Wars primarily took place across *...,If you're referring to the Greco-Persian Wars ...,# The Persian Wars\n\nThe Persian Wars (5th ce...,The Greco-Persian Wars were a series of confli...,The **Persian Wars** (also called the **Greco-...
2,what are add ons,open_qa,"Plug-in (computing) , a piece of software whic...",An **add-on** is something extra or additional...,"""Add-ons"" (sometimes written ""add ons"", ""addon...",# Add-ons\n\nAdd-ons are **additional features...,**Add-ons** are extra software components or e...,An **add-on** is an optional component that en...,"# Add-ons\n\n**Add-ons** (also called ""extensi...",# Add-ons\n\nAdd-ons are **supplementary featu...,# Extensions and Enhancements for Software and...,**Add-ons** are supplementary software compone...
3,how does a dredge work?,open_qa,Dredging is an excavation activity or operatio...,A dredge is essentially a floating machine des...,"A dredge is a machine used to remove sediment,...",# How a Dredge Works\n\nA dredge is a vessel o...,A **dredge** is a machine or system designed t...,A dredge is essentially a floating machine des...,"# Dredging: Equipment, Methods, and Operations...",# How Dredging Works\n\nDredging is the proces...,Dredging is an excavation operation typically ...,A **dredge** is a machine or system designed t...
4,what classes are considered humanities,open_qa,The humanities are academic disciplines that s...,The humanities are academic disciplines that s...,Humanities are academic disciplines that study...,# Humanities Classes\n\nHumanities typically i...,“Humanities” is the broad family of discipline...,The humanities are academic disciplines that s...,# Humanities: An Overview\n\nThe humanities ar...,# Humanities Classes\n\nHumanities courses exa...,The humanities are academic disciplines that s...,"""Humanities"" encompasses the broad family of d..."


### Read the files with GPT5-mini rewritings:

In [4]:
# Reading 5 files which the original writers are 4 LLMs and 1 human
print('Reading dataframes that rewrite Gemini/GPT/Claude/human/Kimi:')
df_gemini = pd.read_csv(gemini_rw_file)
print('\t\tDone! /', end = ' ')
df_gpt = pd.read_csv(gpt_rw_file)
print('Done! /', end = ' ')
df_claude = pd.read_csv(claude_rw_file)
print('Done! /', end = ' ')
df_human = pd.read_csv(human_rw_file)
print('Done! /', end = ' ')
df_kimi = pd.read_csv(kimi_rw_file)
print('Done!')

Reading dataframes that rewrite Gemini/GPT/Claude/human/Kimi:
		Done! / Done! / Done! / Done! / Done!


### Check shapes and merge GPT rewritings with `OriginalWriter_to_Rewriter` column names

In [5]:
# Check shapes
print(f'Gemini shape: {df_gemini.shape}')
print(f'GPT shape: {df_gpt.shape}')
print(f'Claude shape: {df_claude.shape}')
print(f'Human shape: {df_human.shape}')
print(f'Kimi shape: {df_kimi.shape}', end = '\n\n')

# Merge all GPT rewrittings
df_all_claude_and_gpt = df_all_claude.copy()
df_all_claude_and_gpt['gemini_to_gpt'] = df_gemini['gpt5mini_gemini']
df_all_claude_and_gpt['gpt_to_gpt'] = df_gpt['gpt5mini_gpt']
df_all_claude_and_gpt['claude_to_gpt'] = df_claude['gpt5mini_claude']
df_all_claude_and_gpt['human_to_gpt'] = df_human['gpt5mini_human']
df_all_claude_and_gpt['kimi_to_gpt'] = df_kimi['gpt5mini_kimi']

# Print the head
print(f'The merged file contains {len(df_all_claude_and_gpt)} records/rows.')
print(f'Column names: {list(df_all_claude_and_gpt.columns)}')
print('\nThe first 2 rows:')
display(df_all_claude_and_gpt.head(2))

Gemini shape: (5306, 8)
GPT shape: (5306, 8)
Claude shape: (5306, 8)
Human shape: (5306, 8)
Kimi shape: (5306, 8)

The merged file contains 5306 records/rows.
Column names: ['question', 'source', 'human_answers', 'gemini_answers', 'gpt5mini_answer', 'claude_haiku_answers', 'kimi_k2_answers', 'gemini_to_claude', 'gpt_to_claude', 'claude_to_claude', 'human_to_claude', 'kimi_to_claude', 'gemini_to_gpt', 'gpt_to_gpt', 'claude_to_gpt', 'human_to_gpt', 'kimi_to_gpt']

The first 2 rows:


,question,source,human_answers,gemini_answers,gpt5mini_answer,claude_haiku_answers,kimi_k2_answers,gemini_to_claude,gpt_to_claude,claude_to_claude,human_to_claude,kimi_to_claude,gemini_to_gpt,gpt_to_gpt,claude_to_gpt,human_to_gpt,kimi_to_gpt
0,what composer used sound mass,open_qa,"Composers and works include Barbara Kolb , Pau...",The most prominent composer strongly associate...,"Several 20th-century composers used the ""sound...",# Composers Who Used Sound Mass\n\n**Gyorgy Li...,"The term **""sound mass""** (or **""sound-mass co...",The most prominent composer associated with th...,"Several 20th-century composers pioneered the ""...",# Composers Who Used Sound Mass\n\n**György Li...,Composers employing sound mass techniques incl...,"The term **""sound mass""** (or **""sound-mass co...",The composer most strongly associated with the...,"Several 20th-century composers employed the ""s...",Composers Who Used Sound Mass\n\nGyörgy Ligeti...,Composers and works include Barbara Kolb; Paul...,"The term ""sound mass"" (or ""sound‑mass composit..."
1,where did the persian war take place,open_qa,The Greco-Persian Wars (also often called the ...,The Persian Wars primarily took place in **mai...,If you mean the Greco–Persian Wars (often call...,# The Persian Wars\n\nThe Persian Wars took pl...,The **Persian Wars** (also called the **Greco-...,The Persian Wars primarily took place across *...,If you're referring to the Greco-Persian Wars ...,# The Persian Wars\n\nThe Persian Wars (5th ce...,The Greco-Persian Wars were a series of confli...,The **Persian Wars** (also called the **Greco-...,The Persian Wars were fought mainly in mainlan...,If you mean the Greco–Persian Wars (commonly c...,The Persian Wars (5th century BCE) were fought...,"The Greco-Persian Wars, often called the Persi...",The Persian Wars (also called the Greco-Persia...


### Standardize the order and column namings

- `OriginalWriter_answers` for original writers
- `OriginalWriter_to_Rewriter` for the rewritings

In [6]:
# Define the new order
column_order = [
    'question', 'source',
    'gemini_answers', 'gpt5mini_answer', 'claude_haiku_answers', 'human_answers', 'kimi_k2_answers',
    'gemini_to_claude', 'gpt_to_claude', 'claude_to_claude', 'human_to_claude', 'kimi_to_claude',
    'gemini_to_gpt', 'gpt_to_gpt', 'claude_to_gpt', 'human_to_gpt', 'kimi_to_gpt'
]
df_all_claude_and_gpt = df_all_claude_and_gpt[column_order]

# Rename 'gpt5mini_answer' to 'gpt5mini_answers'
df_all_claude_and_gpt = df_all_claude_and_gpt.rename(columns = {'gpt5mini_answer': 'gpt_answers',
                                                                'claude_haiku_answers': 'claude_answers',
                                                                'kimi_k2_answers': 'kimi_answers'})

# Check results
print(f'The updated dataframe contains {len(df_all_claude_and_gpt)} records/rows.')
print(f'Column names: {list(df_all_claude_and_gpt.columns)}')

print('\nThe first 2 rows:')
display(df_all_claude_and_gpt.head(2))

The updated dataframe contains 5306 records/rows.
Column names: ['question', 'source', 'gemini_answers', 'gpt_answers', 'claude_answers', 'human_answers', 'kimi_answers', 'gemini_to_claude', 'gpt_to_claude', 'claude_to_claude', 'human_to_claude', 'kimi_to_claude', 'gemini_to_gpt', 'gpt_to_gpt', 'claude_to_gpt', 'human_to_gpt', 'kimi_to_gpt']

The first 2 rows:


,question,source,gemini_answers,gpt_answers,claude_answers,human_answers,kimi_answers,gemini_to_claude,gpt_to_claude,claude_to_claude,human_to_claude,kimi_to_claude,gemini_to_gpt,gpt_to_gpt,claude_to_gpt,human_to_gpt,kimi_to_gpt
0,what composer used sound mass,open_qa,The most prominent composer strongly associate...,"Several 20th-century composers used the ""sound...",# Composers Who Used Sound Mass\n\n**Gyorgy Li...,"Composers and works include Barbara Kolb , Pau...","The term **""sound mass""** (or **""sound-mass co...",The most prominent composer associated with th...,"Several 20th-century composers pioneered the ""...",# Composers Who Used Sound Mass\n\n**György Li...,Composers employing sound mass techniques incl...,"The term **""sound mass""** (or **""sound-mass co...",The composer most strongly associated with the...,"Several 20th-century composers employed the ""s...",Composers Who Used Sound Mass\n\nGyörgy Ligeti...,Composers and works include Barbara Kolb; Paul...,"The term ""sound mass"" (or ""sound‑mass composit..."
1,where did the persian war take place,open_qa,The Persian Wars primarily took place in **mai...,If you mean the Greco–Persian Wars (often call...,# The Persian Wars\n\nThe Persian Wars took pl...,The Greco-Persian Wars (also often called the ...,The **Persian Wars** (also called the **Greco-...,The Persian Wars primarily took place across *...,If you're referring to the Greco-Persian Wars ...,# The Persian Wars\n\nThe Persian Wars (5th ce...,The Greco-Persian Wars were a series of confli...,The **Persian Wars** (also called the **Greco-...,The Persian Wars were fought mainly in mainlan...,If you mean the Greco–Persian Wars (commonly c...,The Persian Wars (5th century BCE) were fought...,"The Greco-Persian Wars, often called the Persi...",The Persian Wars (also called the Greco-Persia...


### Save the final dataframe to parquet (with both Claude and GPT rewritings)

In [7]:
df_all_claude_and_gpt.to_parquet(output_file)